# Lab 5.1: Classification and Prediction - Advanced Concepts

## 🎯 Learning Objectives

This comprehensive lab covers:
1. **Classification Algorithms**: Decision Trees, Random Forests, Naive Bayes, KNN, Logistic Regression, ANN, SVM
2. **Model Evaluation**: Training/Testing strategies, Cross-validation techniques
3. **Performance Metrics**: Accuracy, Precision, Recall, ROC curves, Loss functions

## 📚 Table of Contents
- Part 1: Decision Trees & Random Forests
- Part 2: Bayesian Networks (Naive Bayes)
- Part 3: K-Nearest Neighbors (KNN)
- Part 4: Linear & Logistic Regression
- Part 5: Artificial Neural Networks
- Part 6: Support Vector Machines
- Part 7: Model Evaluation Techniques
- Part 8: Performance Metrics & Visualization

In [ ]:
# Import all necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_breast_cancer, load_iris
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, 
    LeaveOneOut, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score,
    log_loss, mean_squared_error, mean_absolute_error
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

## Data Preparation

We'll use the **Breast Cancer Wisconsin** dataset - a classic binary classification problem.

In [ ]:
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Standardize features (important for some algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset: {data.filename}")
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Classes: {data.target_names}")
print(f"Training set: {X_train.shape[0]}, Test set: {X_test.shape[0]}")

# Part 1: Decision Trees & Random Forests

## Decision Trees

**Key Concepts:**
- **Recursive Partitioning**: Splits data based on feature values
- **Impurity Measures**: Gini, Entropy (Information Gain)
- **Pruning**: Prevents overfitting by limiting tree depth
- **Interpretability**: Easy to visualize and understand

**Advantages:**
- Simple to understand and interpret
- Handles both numerical and categorical data
- Requires little data preprocessing

**Disadvantages:**
- Prone to overfitting
- Unstable (small changes in data → different tree)
- Biased toward features with more levels

In [ ]:
# Train Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

# Predictions
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

# Evaluate
print("Decision Tree Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_dt):.4f}")

# Visualize tree (first 3 levels)
plt.figure(figsize=(20, 10))
plot_tree(dt, max_depth=3, feature_names=data.feature_names, 
          class_names=data.target_names, filled=True, fontsize=10)
plt.title("Decision Tree Visualization (Depth=3)", fontsize=16)
plt.show()

## Random Forests

**Key Concepts:**
- **Ensemble Learning**: Combines multiple decision trees
- **Bootstrap Aggregating (Bagging)**: Each tree trained on random sample
- **Feature Randomness**: Random subset of features at each split
- **Voting**: Final prediction by majority vote

**Why Random Forests Work Better:**
1. **Variance Reduction**: Averaging reduces overfitting
2. **Robustness**: Less sensitive to noise
3. **Feature Importance**: Provides reliable importance scores

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Evaluate
print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")

# Feature Importance
feature_imp = pd.DataFrame({
    'feature': data.feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_imp, x='importance', y='feature', palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest)', fontsize=14)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# Part 2: Bayesian Networks (Naive Bayes)

## Naive Bayes Classifier

**Key Concepts:**
- **Bayes' Theorem**: P(A|B) = P(B|A) × P(A) / P(B)
- **Conditional Independence**: Assumes features are independent given class
- **Probabilistic Output**: Provides class probabilities

**Types:**
- **Gaussian NB**: For continuous features (assumes normal distribution)
- **Multinomial NB**: For discrete counts (text classification)
- **Bernoulli NB**: For binary features

**Advantages:**
- Fast training and prediction
- Works well with small datasets
- Handles high-dimensional data

**Disadvantages:**
- Independence assumption often violated
- Poor probability estimates (but good classification)

In [ ]:
# Train Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)

# Predictions
y_pred_nb = nb.predict(X_test)
y_prob_nb = nb.predict_proba(X_test)[:, 1]

# Evaluate
print("Naive Bayes Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_nb):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_nb):.4f}")
print(f"Log Loss: {log_loss(y_test, nb.predict_proba(X_test)):.4f}")

# Part 3: K-Nearest Neighbors (KNN)

**Key Concepts:**
- **Instance-Based Learning**: No explicit training phase
- **Distance Metrics**: Euclidean, Manhattan, Minkowski
- **K Parameter**: Number of neighbors to consider
- **Voting**: Majority vote for classification

**How it Works:**
1. Calculate distance to all training samples
2. Select K nearest neighbors
3. Assign class by majority vote

**Choosing K:**
- Small K: More sensitive to noise (high variance)
- Large K: Smoother boundaries (high bias)
- Odd K: Avoids ties in binary classification

In [ ]:
# Find optimal K
k_values = range(1, 31)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    accuracies.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

# Plot K vs Accuracy
plt.figure(figsize=(10, 6))
plt.plot(k_values, accuracies, marker='o', linewidth=2)
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN: Choosing Optimal K', fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Train with optimal K
optimal_k = k_values[np.argmax(accuracies)]
knn = KNeighborsClassifier(n_neighbors=optimal_k)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:, 1]

print(f"Optimal K: {optimal_k}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_knn):.4f}")

# Part 4: Linear & Logistic Regression

## Logistic Regression

**Key Concepts:**
- **Sigmoid Function**: Maps linear output to [0,1]
- **Log-Odds**: ln(p/(1-p)) = β₀ + β₁x₁ + ... + βₙxₙ
- **Maximum Likelihood**: Optimizes probability of observed data

**Advantages:**
- Provides probability estimates
- Efficient and interpretable
- Works well for linearly separable data

**Regularization:**
- **L1 (Lasso)**: Feature selection
- **L2 (Ridge)**: Prevents overfitting

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=10000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")

# Coefficient analysis
coef_df = pd.DataFrame({
    'feature': data.feature_names,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df, x='coefficient', y='feature', palette='coolwarm')
plt.title('Top 10 Logistic Regression Coefficients', fontsize=14)
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

# Part 5: Artificial Neural Networks (ANN)

**Key Concepts:**
- **Layers**: Input → Hidden → Output
- **Activation Functions**: ReLU, Sigmoid, Tanh
- **Backpropagation**: Gradient descent for weight updates
- **Epochs**: Number of passes through training data

**Architecture Choices:**
- **Number of hidden layers**: Depth
- **Neurons per layer**: Width
- **Activation function**: Non-linearity
- **Learning rate**: Step size for optimization

In [ ]:
# Train Neural Network
ann = MLPClassifier(
    hidden_layer_sizes=(100, 50),  # 2 hidden layers
    activation='relu',
    max_iter=500,
    random_state=42
)
ann.fit(X_train_scaled, y_train)

y_pred_ann = ann.predict(X_test_scaled)
y_prob_ann = ann.predict_proba(X_test_scaled)[:, 1]

print("Neural Network Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_ann):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_ann):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_ann):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_ann):.4f}")
print(f"\nTraining iterations: {ann.n_iter_}")
print(f"Final loss: {ann.loss_:.4f}")

# Part 6: Support Vector Machines (SVM)

**Key Concepts:**
- **Maximum Margin**: Finds optimal separating hyperplane
- **Support Vectors**: Critical points defining the boundary
- **Kernel Trick**: Maps data to higher dimensions
  - Linear, Polynomial, RBF (Radial Basis Function)
- **C Parameter**: Trade-off between margin and misclassification

**When to Use:**
- High-dimensional data
- Clear margin of separation
- More features than samples

In [ ]:
# Train SVM with RBF kernel
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

print("SVM Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_svm):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_svm):.4f}")
print(f"\nNumber of support vectors: {svm.n_support_}")

# Part 7: Model Evaluation Techniques

## Training and Testing Strategies

### 1. Holdout Method
- Split data into train/test sets
- Typical split: 70/30 or 80/20
- **Pros**: Fast, simple
- **Cons**: High variance, wastes data

### 2. K-Fold Cross-Validation
- Split data into K folds
- Train on K-1 folds, test on remaining
- Repeat K times, average results
- **Pros**: Better use of data, lower variance
- **Cons**: K times slower

### 3. Stratified K-Fold
- Maintains class distribution in each fold
- Important for imbalanced datasets

### 4. Leave-One-Out Cross-Validation (LOOCV)
- K = number of samples
- **Pros**: Maximum data usage
- **Cons**: Very slow, high variance

### 5. Bootstrap
- Random sampling with replacement
- Creates multiple training sets

In [ ]:
# Compare different CV strategies
models = {
    'Decision Tree': dt,
    'Random Forest': rf,
    'Naive Bayes': nb,
    'KNN': knn,
    'Logistic Regression': lr,
    'Neural Network': ann,
    'SVM': svm
}

cv_results = []

# 5-Fold Cross-Validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    # Use scaled data for models that need it
    X_cv = X_train_scaled if name in ['KNN', 'Logistic Regression', 'Neural Network', 'SVM'] else X_train
    
    scores = cross_val_score(model, X_cv, y_train, cv=kfold, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'Mean CV Score': scores.mean(),
        'Std CV Score': scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values('Mean CV Score', ascending=False)
print("\n5-Fold Cross-Validation Results:")
print(cv_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(cv_df['Model'], cv_df['Mean CV Score'], xerr=cv_df['Std CV Score'], 
         capsize=5, alpha=0.7, color='steelblue')
plt.xlabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Model Comparison: 5-Fold CV', fontsize=14)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Part 8: Performance Metrics & Visualization

## Confusion Matrix

```
                Predicted
              Neg    Pos
Actual  Neg   TN     FP
        Pos   FN     TP
```

## Metrics

- **Accuracy** = (TP + TN) / Total
- **Precision** = TP / (TP + FP) - "How many predicted positives are correct?"
- **Recall (Sensitivity)** = TP / (TP + FN) - "How many actual positives did we find?"
- **F1-Score** = 2 × (Precision × Recall) / (Precision + Recall)
- **Specificity** = TN / (TN + FP)

## ROC Curve
- Plots True Positive Rate vs False Positive Rate
- AUC (Area Under Curve): Overall performance measure
- Perfect classifier: AUC = 1.0
- Random classifier: AUC = 0.5

## Precision-Recall Curve
- Better for imbalanced datasets
- Shows trade-off between precision and recall

In [ ]:
# Create confusion matrices for all models
predictions = {
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf,
    'Naive Bayes': y_pred_nb,
    'KNN': y_pred_knn,
    'Logistic Regression': y_pred_lr,
    'Neural Network': y_pred_ann,
    'SVM': y_pred_svm
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=data.target_names, yticklabels=data.target_names)
    axes[idx].set_title(name, fontsize=12)
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

axes[-1].axis('off')  # Hide last subplot
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
probabilities = {
    'Decision Tree': y_prob_dt,
    'Random Forest': y_prob_rf,
    'Naive Bayes': y_prob_nb,
    'KNN': y_prob_knn,
    'Logistic Regression': y_prob_lr,
    'Neural Network': y_prob_ann,
    'SVM': y_prob_svm
}

plt.figure(figsize=(10, 8))

for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Curves
plt.figure(figsize=(10, 8))

for name, y_prob in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)
    plt.plot(recall, precision, linewidth=2, 
             label=f'{name} (AP = {avg_precision:.3f})')

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves', fontsize=14)
plt.legend(loc='lower left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Summary & Model Comparison

## Final Performance Comparison

In [ ]:
# Comprehensive comparison
final_results = []

for name, y_pred in predictions.items():
    y_prob = probabilities[name]
    final_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(final_results).sort_values('F1-Score', ascending=False)
print("\nFinal Model Comparison:")
print(results_df.to_string(index=False))

# Visualize all metrics
fig, ax = plt.subplots(figsize=(12, 6))
results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', ax=ax, rot=45, width=0.8
)
plt.ylabel('Score', fontsize=12)
plt.title('Comprehensive Model Performance Comparison', fontsize=14)
plt.legend(loc='lower right')
plt.ylim([0.85, 1.0])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Key Takeaways

## Algorithm Selection Guide

| Algorithm | Best For | Avoid When |
|-----------|----------|------------|
| **Decision Tree** | Interpretability, mixed data types | Need high accuracy, unstable data |
| **Random Forest** | High accuracy, feature importance | Need interpretability, real-time prediction |
| **Naive Bayes** | Text classification, fast training | Features are dependent |
| **KNN** | Non-linear boundaries, simple baseline | Large datasets, high dimensions |
| **Logistic Regression** | Probability estimates, interpretability | Non-linear relationships |
| **Neural Networks** | Complex patterns, large datasets | Small datasets, need interpretability |
| **SVM** | High-dimensional data, clear margins | Very large datasets, need probabilities |

## Evaluation Strategy

1. **Always use cross-validation** for robust estimates
2. **Choose metrics based on problem**:
   - Balanced classes → Accuracy
   - Imbalanced classes → F1-Score, ROC-AUC
   - Cost-sensitive → Precision (minimize FP) or Recall (minimize FN)
3. **Visualize performance** with ROC and PR curves
4. **Consider multiple metrics** - no single metric tells the whole story

## Best Practices

- **Feature scaling**: Essential for KNN, SVM, Neural Networks, Logistic Regression
- **Hyperparameter tuning**: Use GridSearchCV or RandomizedSearchCV
- **Ensemble methods**: Often outperform single models
- **Domain knowledge**: Understand your data and problem context